### Experimental setup and adjustments

- **Sizes:** operations use n = 1,000 / 5,000 / 10,000 / 25,000. Insertion sort uses n = 1,000 / 2,000 / 4,000 / 8,000 instead, because pure-Python insertion sort is O(n²) and 25,000 elements would take about a minute per run (6 input/structure combinations × 7 runs).
- **Timing:** `time.perf_counter()`, median of 7 runs. Garbage collection is disabled during each timed region. Very fast operations (read middle, binary search) are repeated inside the timed region and divided, to beat timer resolution.
- **Same logical data:** both structures are built from the same list of `MaintenanceRequest` objects (IDs 1000 to 1000+n-1).
- **Setup is untimed and bypasses `append`:** `append`/`push`/`insertAt` run an O(n) duplicate check, so building 25,000 items through them would cost about 300 million comparisons. Lists are built directly, and only the operation under test is timed.
- **Spare capacity:** the array is built with capacity 2n so that no resize lands inside a timed insert.
- **Sort input:** priorities are distinct (0 to n-1) so sorted and reverse inputs are clean best and worst cases. With the real 1 to 4 scale, nearly everything would be a tie at large n.

In [1]:
import time, random, statistics, gc
import pandas as pd
import matplotlib.pyplot as plt

SIZES = [1_000, 5_000, 10_000, 25_000]
SORT_SIZES = [1_000, 2_000, 4_000, 8_000]
RUNS = 7
NEW_ID = 10**9
random.seed(42)

def make_requests(n):

    return [MaintenanceRequest(1000 + i, "Smith Hall", "plumbing", 1 + i % 4, i, "open")
            for i in range(n)]

def new_req():
    return MaintenanceRequest(NEW_ID, "Smith Hall", "hvac", 1, 0, "open")

def build_array(reqs, cls=ArrayRequestList):
    a = cls(max(1, 2 * len(reqs)))
    for i, r in enumerate(reqs):
        a.data[i] = r
    a.size = len(reqs)
    return a

def build_linked(reqs):
    lst, prev = LinkedRequestList(), None
    for r in reqs:
        node = Node(r)
        if prev is None: lst.head = node
        else: prev.next = node
        prev = node
    lst.tail, lst.size = prev, len(reqs)
    return lst

def bench(setup, op, runs=RUNS, inner=1):
    """Median seconds per call. Setup is untimed; `inner` repeats non-mutating ops to beat timer noise."""
    times = []
    for _ in range(runs):
        obj = setup()
        gc.collect(); gc.disable()
        t0 = time.perf_counter()
        for _ in range(inner):
            op(obj)
        t1 = time.perf_counter()
        gc.enable()
        times.append((t1 - t0) / inner)
    return statistics.median(times)

def operations(n):
    reqs = make_requests(n)
    mid, mid_id = n // 2, 1000 + n // 2
    A = lambda: build_array(reqs)
    L = lambda: build_linked(reqs)
    S = lambda: build_array(reqs, SortedArrayRequestList)

    return {
        ("Read middle", "Array"):          (A, lambda x: x.getAt(mid), 1000),
        ("Read middle", "Linked"):         (L, lambda x: x.getAt(mid), 5),
        ("Insert at beginning", "Array"):  (A, lambda x: x.push(new_req()), 1),
        ("Insert at beginning", "Linked"): (L, lambda x: x.push(new_req()), 1),
        ("Append", "Array"):               (A, lambda x: x.append(new_req()), 1),
        ("Append", "Linked"):              (L, lambda x: x.append(new_req()), 1),
        ("Insert in middle", "Array"):     (A, lambda x: x.insertAt(mid, new_req()), 1),
        ("Insert in middle", "Linked"):    (L, lambda x: x.insertAt(mid, new_req()), 1),
        ("Remove from beginning", "Array"):  (A, lambda x: x.removeAt(0), 1),
        ("Remove from beginning", "Linked"): (L, lambda x: x.removeAt(0), 1),
        ("Remove from middle", "Array"):   (A, lambda x: x.removeAt(mid), 1),
        ("Remove from middle", "Linked"):  (L, lambda x: x.removeAt(mid), 1),
        ("Search for missing ID", "Array"):    (A, lambda x: x.findById(-1), 1),
        ("Search for missing ID", "Linked"):   (L, lambda x: x.findById(-1), 1),
        ("Linear search", "Array"):        (A, lambda x: x.linearSearchById(mid_id), 1),
        ("Linear search", "Linked"):       (L, lambda x: x.linearSearchById(mid_id), 1),
        ("Binary search (missing)", "Array"): (S, lambda x: x.binarySearchById(-1), 1000),
    }

rows = []
for n in SIZES:
    for (name, struct), (setup, op, inner) in operations(n).items():
        rows.append(dict(n=n, operation=name, structure=struct, seconds=bench(setup, op, inner=inner)))
ops_df = pd.DataFrame(rows)
ops_df.pivot_table(index=["operation", "structure"], columns="n", values="seconds")

NameError: name 'ArrayRequestList' is not defined

In [ ]:
def sort_requests(n, kind):
    pr = list(range(n))
    if kind == "reverse": pr.reverse()
    elif kind == "random": random.Random(1).shuffle(pr)
    return [MaintenanceRequest(i, "Smith Hall", "plumbing", p, i, "open") for i, p in enumerate(pr)]


chk = build_linked(sort_requests(200, "random")); insertionSortLinked(None, chk)
assert [chk.getAt(i).priority for i in range(200)] == list(range(200))
chk = build_array(sort_requests(200, "random")); insertionSortArray(None, chk)
assert [chk.getAt(i).priority for i in range(200)] == list(range(200))

sort_rows = []
for n in SORT_SIZES:
    for kind in ("sorted", "reverse", "random"):
        reqs = sort_requests(n, kind)
        sort_rows.append(dict(n=n, input=kind, structure="Array",
            seconds=bench(lambda: build_array(reqs), lambda x: insertionSortArray(None, x))))
        sort_rows.append(dict(n=n, input=kind, structure="Linked",
            seconds=bench(lambda: build_linked(reqs), lambda x: insertionSortLinked(None, x))))
sort_df = pd.DataFrame(sort_rows)

sort_df.pivot_table(index=["input", "structure"], columns="n", values="seconds")

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, struct in zip(axes, ("Array", "Linked")):
    for name, g in ops_df[ops_df.structure == struct].groupby("operation"):
        ax.plot(g.n, g.seconds, marker="o", label=name)
    ref = ops_df[(ops_df.structure == struct) & (ops_df.n == SIZES[0])].seconds.median()
    ax.plot(SIZES, [ref * n / SIZES[0] for n in SIZES], "k--", label="reference O(n)")
    ax.set(xscale="log", yscale="log", title=f"{struct}: time per operation", xlabel="n")
axes[0].set_ylabel("median seconds")
axes[1].legend(fontsize=7); plt.tight_layout(); plt.show()


fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, struct in zip(axes, ("Array", "Linked")):
    for kind, g in sort_df[sort_df.structure == struct].groupby("input"):
        ax.plot(g.n, g.seconds, marker="o", label=kind)
    ax.set(xscale="log", yscale="log", title=f"Insertion sort ({struct})", xlabel="n")
axes[0].set_ylabel("median seconds"); axes[0].legend(); plt.tight_layout(); plt.show()


fig, ax = plt.subplots(figsize=(6, 4))
for name in ("Linear search", "Binary search (missing)"):
    g = ops_df[(ops_df.operation == name) & (ops_df.structure == "Array")]
    ax.plot(g.n, g.seconds, marker="o", label=name)
ax.set(xscale="log", yscale="log", title="Array: linear vs binary search", xlabel="n", ylabel="median seconds")
ax.legend(); plt.tight_layout(); plt.show()


piv = ops_df.pivot_table(index=["operation", "structure"], columns="n", values="seconds")
piv["25k / 5k"] = piv[25_000] / piv[5_000]
spv = sort_df.pivot_table(index=["input", "structure"], columns="n", values="seconds")
spv["8k / 2k"] = spv[8_000] / spv[2_000]
display(piv[["25k / 5k"]].round(2)); display(spv[["8k / 2k"]].round(2))

## Observed vs. predicted

**Result that differed from my prediction:** (describe one specific result, with the numbers from the ratio table)

**Why:** (e.g., `append`/`push`/`insertAt` run an O(n) duplicate check before the O(1) work, so they scale linearly in both structures)

**Was the Big-O analysis still correct?** (Yes or no, and explain)

## Benchmarks vs. Big-O

Timing shows what happened on this machine, with this interpreter, at these sizes. Big-O explains how the amount of work grows with n. Constant factors (Python loops, pointer chasing, cache behavior) can reorder the structures at small n without changing the growth rate. The two are complementary, and neither replaces the other.

<h2> 11. Part H — Complexity Analysis

**1. Dynamic Array vs. Linked List**

| **Operation** | **Dynamic Array** | **Linked List**| **Explanation**|
|---|---|---|---|
|**Indexed access getAt()** | O(1)) | O(n) | Array reads the index directly. Linked list follows nodes from head to target. |
|**Search by ID (findById)** | O(n) | O(n) | Both compare IDs one by one. A missing ID requires checking all requests|
|**Push / prepend (push)** | O(n) | O(n) | Array checks duplicated and shifts requests right. Linked list checks duplicates, then updates head|
|**Append (append)** | O(n) | O(n) | Both scan for duplicate IDs. Array may resize. Linked list attached directly to end using tail.|
|**Insert at position (insertAt)** | O(n) | O(n) | Array checks for duplicated, shifts requests, and may resize. Linked list checks duplicates, traverses, then changes links.|
|**Remove at position (removeAt)** | O(n) | O(n) | Array shifts later requests left. Linked list traverses to the previous node, then updates the links|
|**Count (count)** | O(1) | O(1) | Both return the stored self.size. No traversal or counting loop needed|


**2. Searching and Sorting Algorithms**

|**Algorithm**| **Worst-Case Time** | **Extra Space** | **Explanation**|
|---|---|---|---|
|**Linear Search** | O(n) | O(1) | Checks IDs one by one and may examine all requests. Uses only a few variables to track the search. |
|**Binary Search** | O(log n) | O(1) | Checks the middle of a sorted array and cuts the search area in half each time. No extra array is needed. |
|**First-occurence Search** | O(log n) |O(1) |Uses binary search on sorted priorities. Keeps checking left to find the first match.|
|**Insertion Sort (array)**| O(n²) | O(1) | Compares and shifts requests until they are in priority order. Sorts within the same array.|
|**Insertion Sort (linked list)**| O(n²) | O(1) | Finds the correct place for each node and changes the links. No new list is created. |
|**Merge Sort (array)**| O(n log n) | O(n) | Splits the requetss into smaller groups, then merges them in time order. Uses a temporary array.|
|**Merge Sort (linked list)**| O(n log n) | O(log n) | Splits the linked list and merges nodes in time order. Uses memory for recursive calls. |


**3. Insertion Sort - Best, Average, and Worst Cases**

|**Structure**| **Case** | **Time**| **Explanation**|
|---|---|---|---|
|**Array** | Best | O(n) | Already sorted at start. Each request is checked, but nothing needs shifting. |
|**Array** | Average | O(n²) | Mixed order at the start. Some requests need several comparisons and shifts.|
|**Array** | Worst | O(n²) | Reverse order at start. Each request must be shifted before earlier requests. |
|**Linked List**| Best | O(n) | Reverse order at start, Each node is added to the front of the chain. |
|**Linked List**| Average | O(n²) | Mixed order at start. Find each nodes position requires scanning the chain. |
|**Linked List**| Worst | O(n²) | Already sorted at start. Each node must be placed at the end of the growing sorted chain.|


**4. Additional Cost Explanations**

Some methods perform more than one operation. Our group considered each cost seperately when deciding the overall time complexity. Here is the breakdown of this:

*   **Duplicate ID checks:** Both the array and the linked list check for cuplicate IDs before adding a request. This requires searching through up to n requests, taking O(n) time. Because of this, even append() takes O(n) time in our implmentations.

*   **Array Shifting:** The arrays push() and insertAt() may shift requests to make more room. removeAt() may shift requests left to close the gap. Shifting can take O(n) time.

*   **Linked list traveral and link changes:** This linked list may need to walk through nodes to reach a position, taking O(n) time. Once the correct position is found, changing the links takes O(1) time. Traveral is teh larher cost in this case.

*   **Array resizing:** When the array is full, _resize() creates a larger array and copies the existing requests. This takes O(n) time and O(n) extra space. When resiizing is not needed, insertion would use O(1) extra space instead.

*   **Searching:** Linear search checks requests one by one, taking O(n) time. Binary search cuts the search range roughly in half each step, taking O(log n) time when the array is sorted.

*   **Sorting:** Insertion sort can take O(n²) time because requests may need repeated comparisons, shifts, or node traversal. Merge sort takes O(n log n) time because it repeatedly splits and merges the requests. Array merge sorts uses O(n) temporary space, while linked list merge sort uses O(log n) space for recursive calling.

*   **Overall complexity - keep dominant term:** When a method has multiple costs, we add them and keep the largest growing term. For example, an insertion that checks duplicates, shifts requests, and stores the new request has a time complexity like this:

O(n) + O(n) + O(1). We need O(n) as the dominant term because they grow with the number or requests.












